# Re-run Fase 6 (NFSP) amb criteri corregit — Google Colab

Executa el re-run del `best_nash` amb el fix (llindar 10M passos + 60 partides balancejades),
aprofitant la **GPU de Colab** (a Windows local no es pot per `fork` + manca de GPU).

**Abans de començar:**
1. Activa GPU: *Entorn d'execució → Canviar tipus d'entorn → T4 GPU*.
2. Assegura't que el fix d'`entrenament_fase6.py` està **pujat a GitHub** (commit + push).
3. Puja a Google Drive (carpeta `TFG_rerun/`) els dos pesos:
   - `best_pesos_cos_truc.pth`
   - `best.zip` (el de F6: `TFG_Doc/notebooks/6_nfsp/resultats/ppo_nfsp/best.zip`)

## 1. Comprovar GPU

In [ ]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CAP — activa GPU a Entorn d execucio!')

## 2. Clonar el repo (codi)

In [ ]:
%cd /content
![ -d TFG-truc ] && rm -rf TFG-truc
!git clone https://github.com/JoFeF08/TFG-truc.git
%cd /content/TFG-truc
# Verifica que el fix hi és (ha de sortir NASH_MIN_STEPS):
!grep -n 'NASH_MIN_STEPS\|t >= nash_min_steps' RL/entrenament/entrenamentsComparatius/fase6/entrenament_fase6.py

## 3. Instal·lar dependències

In [ ]:
!pip install -q rlcard stable-baselines3 sb3-contrib gymnasium tqdm
print('Dependencies OK')

## 4. Muntar Drive i recollir els pesos

Abans, puja a Drive `MyDrive/TFG_rerun/best_pesos_cos_truc.pth` i `MyDrive/TFG_rerun/best.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
SRC = '/content/drive/MyDrive/TFG_rerun'
PESOS_COS = '/content/best_pesos_cos_truc.pth'
MODEL_INI = '/content/best.zip'
shutil.copy(f'{SRC}/best_pesos_cos_truc.pth', PESOS_COS)
shutil.copy(f'{SRC}/best.zip', MODEL_INI)
print('pesos_cos :', os.path.getsize(PESOS_COS), 'bytes')
print('model_ini :', os.path.getsize(MODEL_INI), 'bytes')

# Carpeta de sortida PERSISTENT a Drive (sobreviu a desconnexions)
SAVE_DIR = '/content/drive/MyDrive/TFG_rerun/ppo_nfsp_rerun'
os.makedirs(SAVE_DIR, exist_ok=True)
print('save_dir  :', SAVE_DIR)

## 5. Llançar el re-run

Els defaults ja apliquen el fix (`NASH_MIN_STEPS=10M`, `sl_eval_sessions=60`). `--steps 25000000`
és el pressupost recomanat; si vols una prova ràpida primer, baixa'l a `2000000`.
Els resultats es desen a Drive, així que una desconnexió no els perd.

In [ ]:
%cd /content/TFG-truc
!python RL/entrenament/entrenamentsComparatius/fase6/entrenament_fase6.py \
  --pesos_cos /content/best_pesos_cos_truc.pth \
  --model_inicial /content/best.zip \
  --steps 25000000 \
  --num_envs 8 \
  --save_dir /content/drive/MyDrive/TFG_rerun/ppo_nfsp_rerun

## 6. Verificar resultats

El nou `best_nash.zip` (ja amb el criteri corregit) queda a
`MyDrive/TFG_rerun/ppo_nfsp_rerun/`. Baixa'l al teu repo local a
`TFG_Doc/notebooks/6_nfsp/resultats/ppo_nfsp_rerun/` i el notebook `comparacio_checkpoint3.ipynb`
el recollirà automàticament (afegeix la columna `best_nash_rerun`).

In [ ]:
import os
OUT = '/content/drive/MyDrive/TFG_rerun/ppo_nfsp_rerun'
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if os.path.isfile(p):
        print(f'{f:<24} {os.path.getsize(p):>12,} bytes')

# Cop d ull al millor Nash gap del log
import pandas as pd
df = pd.read_csv(os.path.join(OUT, 'training_log.csv'))
df = df[df.step >= 10_000_000]
print('\nMillor exploit_vs_sl (>=10M):', round(df.exploit_vs_sl.min(), 2), 'pp al pas', int(df.loc[df.exploit_vs_sl.idxmin(), 'step']))